# 古文を現代語に訳す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_machine_translation/classical_to_modern.ipynb)

NLP クックブック 24-17 ／ 機械翻訳

古文を現代語に訳したい。LLM に訳させ、品詞分解の情報を与えると訳が良くなるかも試す。

In [1]:
!pip install -q sacrebleu

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def chat(prompt, max_new_tokens=160):
    ids = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0]

# (古文, 参考にする現代語訳) の組
data = [
    ("今は昔、竹取の翁といふ者ありけり。", "今となっては昔のことだが、竹取の翁という者がいた。"),
    ("野山にまじりて竹を取りつつ、よろづのことに使ひけり。", "野山に分け入って竹を取っては、いろいろなことに使っていた。"),
    ("春はあけぼの。やうやう白くなりゆく山ぎは、少し明かりて、紫だちたる雲の細くたなびきたる。", "春は明け方がよい。だんだん白くなっていく山ぎわが少し明るくなって、紫がかった雲が細くたなびいているのがよい。"),
    ("つれづれなるままに、日暮らし、硯に向かひて、心にうつりゆくよしなしごとを、そこはかとなく書きつくれば、あやしうこそものぐるほしけれ。",
     "することもなく退屈なのにまかせて、一日中硯に向かって、心に浮かんでは消えていくとりとめのないことを、何となく書きつけていると、妙に気持ちが高ぶってくる。"),
    ("行く河の流れは絶えずして、しかももとの水にあらず。", "流れていく川の流れは絶えることがなく、それでいて、もとの水ではない。"),
    ("男もすなる日記といふものを、女もしてみむとてするなり。", "男も書くという日記というものを、女の私も書いてみようと思って書くのである。"),
]

PLAIN = "次の古文を、意味を変えずに現代語に訳してください。訳文だけを 1 行で出力してください。\n\n古文: {}"
FEWSHOT = ("古文を現代語に訳します。訳文だけを 1 行で出力してください。\n\n古文: いづれの御時にか、女御、更衣あまたさぶらひたまひけるなかに\n現代語訳: どの帝の御代であったか、女御や更衣が大勢お仕えしていらっしゃった中に\n\n"
           "古文: 月日は百代の過客にして、行きかふ年もまた旅人なり。\n現代語訳: 月日は永遠の旅人であり、過ぎ去ってはやって来る年もまた旅人である。\n\n古文: {}\n現代語訳:")
for src, ref in data:
    print(f"{src}\n   訳 (指示のみ): {chat(PLAIN.format(src))}\n   訳 (例を示す): {chat(FEWSHOT.format(src))}\n   参考: {ref}")

今は昔、竹取の翁といふ者ありけり。
   訳 (指示のみ): 今は昔、竹取の翁という者がいた。
   訳 (例を示す): 今は昔、竹取の翁という者がいた。
   参考: 今となっては昔のことだが、竹取の翁という者がいた。
野山にまじりて竹を取りつつ、よろづのことに使ひけり。
   訳 (指示のみ): 「野や山を巡って竹を採取し、様々なことに使用した」
   訳 (例を示す): どの帝の御代であったろうか、多くの女御や更衣が帝に仕えていた中で、
   参考: 野山に分け入って竹を取っては、いろいろなことに使っていた。
春はあけぼの。やうやう白くなりゆく山ぎは、少し明かりて、紫だちたる雲の細くたなびきたる。
   訳 (指示のみ): 春はあけぼの。徐々に白んでいく山の辺が少し明るくなり、紫色がかった雲が細くたなびいている。
   訳 (例を示す): 春はあけぼのが美しい。徐々に白んでいく山の辺が少し明るくなり、紫色がかった雲が細く棚引いている。
   参考: 春は明け方がよい。だんだん白くなっていく山ぎわが少し明るくなって、紫がかった雲が細くたなびいているのがよい。
つれづれなるままに、日暮らし、硯に向かひて、心にうつりゆくよしなしごとを、そこはかとなく書きつくれば、あやしうこそものぐるほしけれ。
   訳 (指示のみ): 「手持ち無沙汰な状態で一日中、硯に向かって、心に浮かぶとりとめもない thoughts を漫然と書きつけていると、奇妙なほどに熱中してしまうものだ。」
   訳 (例を示す): 「手持ち無沙汰なまま一日中、硯に向かって、心に浮かぶとりとめもない thoughts を漫然と書きつけていると、奇妙なほどに熱中してしまうのだった。」
   参考: することもなく退屈なのにまかせて、一日中硯に向かって、心に浮かんでは消えていくとりとめのないことを、何となく書きつけていると、妙に気持ちが高ぶってくる。
行く河の流れは絶えずして、しかももとの水にあらず。
   訳 (指示のみ): 「流れる川の姿は常に変わるが、その水自体は元のものではない。」
   訳 (例を示す): 「流れる川水は絶えることなく、しかし元の水と同じではない。」
   参考: 流れていく川の流れは絶えることがなく、それでいて、もとの水ではない。
男もすなる日記といふものを、女もしてみむとてするな

In [3]:
import sacrebleu

for label, prompt in [("指示のみ", PLAIN), ("例を示す", FEWSHOT)]:
    hyps = [chat(prompt.format(src)) for src, _ in data]
    print(label, "chrF =", round(sacrebleu.corpus_chrf(hyps, [[ref for _, ref in data]]).score, 1))
print("訳さずにそのまま出した場合 chrF =", round(sacrebleu.corpus_chrf([src for src, _ in data], [[ref for _, ref in data]]).score, 1))

指示のみ chrF = 35.5
例を示す chrF = 33.8
訳さずにそのまま出した場合 chrF = 17.7


In [4]:
# 誤訳しやすい語 (古今異義語) の語釈を与えてから訳させる
GLOSSARY = {"あやし": "不思議だ、妙だ (現代語の「怪しい」とは違う)", "ものぐるほし": "気持ちが高ぶる、正気を失ったようだ", "つれづれなり": "することがなく退屈だ",
            "よしなしごと": "とりとめのないこと", "すなる": "する + 伝聞の「なり」= するという", "やうやう": "だんだん、しだいに"}
src = data[3][0]
notes = "\n".join(f"- {k}: {v}" for k, v in GLOSSARY.items() if k[:3] in src)
print(notes)
print(chat(f"次の古文を現代語に訳してください。語釈を参考にし、訳文だけを 1 行で出力してください。\n\n語釈:\n{notes}\n\n古文: {src}"))

- あやし: 不思議だ、妙だ (現代語の「怪しい」とは違う)
- ものぐるほし: 気持ちが高ぶる、正気を失ったようだ
- つれづれなり: することがなく退屈だ
- よしなしごと: とりとめのないこと
手持ち無沙汰な状態で一日中、硯に向かってとりとめもない thoughts を漫然と書きつけていると、奇妙なほどに気持ちが高ぶってくるものだ。


- 古文から現代語への翻訳は、同じ言語の中での「時代をまたぐ翻訳」。語順はほぼ同じなので、難しさは語彙 (古今異義語)、助動詞の意味、省略された主語や敬語の解釈にある。
- LLM は有名な作品の冒頭なら訳を覚えていることが多い。知られていない文での性能は、それより低いと考える。評価には、モデルが見たことのなさそうな文を使う。
- この実験では、例を示しても訳は良くならなかった (chrF は「指示のみ」が 33.5〜35.5、「例を示す」が 33.8。6 文しかないので、この程度の差は実行する環境による揺れの範囲)。2 文目では、例として示した源氏物語の訳をそのまま出力する失敗も出た。一方、語釈を与えると「ものぐるほし」が正しく「気持ちが高ぶる」と訳された。訳文に英単語 (thoughts) が混ざる失敗も出ている。
- 小さなモデルは、古語を現代語の似た語と取り違える (「あやし」を「怪しい」と訳す)。語釈や品詞分解の結果 (形態素解析の章の古文のレシピ) をプロンプトに入れると、誤訳を減らせることがある。
- 対訳データがあれば、この章のファインチューニングのレシピがそのまま使える。現代語の単語ベクトルを古文に適応させる、古文の単言語データで逆翻訳する、といった工夫が研究されている。
- 参考訳は 1 つの解釈にすぎず、直訳か意訳かでも大きく変わる。自動評価の数値は目安にとどめる。